# Attitude agility — pitch

A worked example of the `quicksat` agility budget about the **pitch** axis, and the companion to [`agility_roll.ipynb`](agility_roll.ipynb). Both the nominal four-wheel case and the one-wheel-failed case are carried through.

The machinery is identical. The wheel pyramid's symmetry axis is along yaw, which puts roll and pitch both in its base plane, so they see the same projection, the same momentum and the same torque. **Only the inertia differs** — and in this sample, only through the appendage factor, because the body is 1.5 × 1.5 m square in x and y and the box term comes out the same for both axes.

That makes this notebook worth reading beside the roll one for a single reason: it is where the two axes disagree, and where the inertia cases disagree about *how* they disagree.

For *why* each step works the way it does, see `docs/agility_ref.ipynb`.

In [1]:
# Change log. LAST_CHANGE and CHANGE_NOTE are typed in by hand: update them whenever
# the inputs move -- a wheel changed, an inertia case remeasured, a settling time
# renegotiated -- so that the figures below can be read against what produced them.
# The run time is recorded automatically, and says how stale the outputs stored in
# this notebook are relative to that last change.
from datetime import datetime

LAST_CHANGE = "2026-09-21"
CHANGE_NOTE = "First pitch agility budget: three inertia cases, 4 wheels at 26.5 deg."

print(f"last change  {LAST_CHANGE}")
print(f"             {CHANGE_NOTE}")
print(f"last run     {datetime.now().astimezone():%Y-%m-%d %H:%M %Z}")

last change  2026-09-21
             First pitch agility budget: three inertia cases, 4 wheels at 26.5 deg.
last run     2026-09-23 23:32 CEST


In [2]:
import os
from pathlib import Path

import pandas as pd

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from sample/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat import u
from quicksat.agility.budget import AgilityBudget, Axis
from quicksat.mass.budget import MassBudget
from quicksat.utils.mission import Mission

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## The inputs

The same three files the roll notebook reads, and the same wheels. The only thing that changes is which axis the inertia is taken about.

The agility config carries **no mass** — the mass budget owns that — and **no target duration**, because how long a manoeuvre may take is asked of a spacecraft rather than being a property of one.

In [3]:
DATA = Path("sample") / "data"
mission = Mission.from_yaml_file(DATA / "mission.yaml")
mass_data = MassBudget.from_csv(DATA / "equipment.csv", DATA / "mass_budget_config.yaml")

pitch = AgilityBudget.from_yaml_file(
    DATA / "agility_config.yaml",
    DATA / "mission.yaml",
    Axis.PITCH,
    "first_guess",
    mass_budget=mass_data,
)

config = pitch.config
print(f"mission       {mission.altitude:~.0f}, ground track {mission.ground_track_speed:~.3f}")
print(f"wheels        {config.wheels.count} at {config.wheels.elevation:~} elevation")
print(f"settling      {config.settling_time:~}")
print(f"inertia cases {', '.join(config.inertia_cases)}")
print()
print(f"flying        '{pitch.case_name}' -- envelope, so the mass comes from the")
print(f"              mass budget at {pitch.case.propellant:~.0f} propellant: {pitch.mass:~.2f}")
print(f"inertia       {pitch.inertia:~.1f}")

mission       500 km, ground track 7.059 km / s
wheels        4 at 26.5 deg elevation
settling      20 s
inertia cases first_guess, measured_bol, measured_eol

flying        'first_guess' -- envelope, so the mass comes from the
              mass budget at 100 % propellant: 470.62 kg
inertia       250.0 kg * m ** 2


## What the wheels can put about the axis

Identical to roll, and that is the point. The pyramid's symmetry axis is along yaw, so every in-plane axis sees the same `cos(elevation)·cos(45°)` fraction of each wheel. Nothing about the wheels knows which axis you are asking about.

In [4]:
print(f"usable per wheel  {pitch.usable_momentum_per_wheel:~.3f}"
      f"  and {pitch.usable_torque_per_wheel:~.3f}")
print(f"projection        {pitch.projection.magnitude:.4f}")
print()
print(f"about pitch        {pitch.axis_momentum():~.3f}  {pitch.axis_torque():~.4f}")
print(f"about yaw         {pitch.yaw_momentum:~.2f}   -- the weak axis under this")
print("                                    mounting; no slew case here, but it is")
print("                                    what a yaw manoeuvre lives within")

usable per wheel  1.332 m * N * s  and 0.150 m * N
projection        0.6328

about pitch        3.372 m * N * s  0.3797 m * N
about yaw         2.38 m * N * s   -- the weak axis under this
                                    mounting; no slew case here, but it is
                                    what a yaw manoeuvre lives within


## The two limits, and where they swap

Momentum sets the fastest the spacecraft can turn; torque sets how quickly it gets there. Below the crossover angle the wheels never saturate and the slew is **torque limited** — a triangular accelerate-then-decelerate profile. Above it they do, and the slew coasts at maximum rate: **momentum limited**, a trapezoid.

Which one binds is the useful output. Torque limited means more torque would buy something; momentum limited means it would not.

In [5]:
print(f"max rate          {pitch.max_rate():~.4f}")
print(f"max acceleration  {pitch.max_acceleration():~.5f}")
print(f"crossover angle   {pitch.crossover_angle():~.2f}")
print()
for angle in (5 * u.deg, 90 * u.deg):
    print(f"  {angle:~4.0f} is {pitch.profile(angle).value:11s} limited by "
          f"{'torque' if angle <= pitch.crossover_angle() else 'momentum'}")

max rate          0.7727 deg / s
max acceleration  0.08701 deg / s ** 2
crossover angle   6.86 deg

     5 deg is triangular  limited by torque
    90 deg is trapezoidal limited by momentum


## How long a slew takes

One row per angle: the slew itself, the slew plus settling, the peak rate reached, and how much of the wheel momentum it called on. Momentum used hits 100% for every slew past the crossover — that is what momentum limited means — and the shortfall below it is headroom a larger slew would spend.

In [6]:
pitch.slew_table()

,angle,profile,slew_time,total_time,peak_rate,momentum_used
0,5,triangular,15.16,35.16,0.66,0.85
1,10,trapezoidal,21.82,41.82,0.77,1.00
2,15,trapezoidal,28.29,48.29,0.77,1.00
3,20,trapezoidal,34.76,54.76,0.77,1.00
4,40,trapezoidal,60.65,80.65,0.77,1.00
5,45,trapezoidal,67.12,87.12,0.77,1.00
6,60,trapezoidal,86.53,106.53,0.77,1.00
7,90,trapezoidal,125.36,145.36,0.77,1.00


## Does it fit?

The target duration is an argument, because the same spacecraft is asked the question differently by different operations. Give one and the budget says how much room is left; ask it the other way round and it says how far it could have turned instead.

A slew costs swath. Tying the time to the shared mission's ground track speed turns it into kilometres of ground given up, which is the number a payload operator actually feels.

In [7]:
angle = 40 * u.deg
for target in (113 * u.s, 90 * u.s):
    margin = pitch.time_margin(angle, target)
    verdict = "fits" if margin.magnitude >= 0 else "DOES NOT FIT"
    print(f"{angle:~.0f} in {target:~.0f}:  needs {pitch.total_time(angle):~.1f}, "
          f"{margin.to('percent'):~+.1f} -- {verdict}")
    print(f"{'':>17}largest slew that would fit: "
          f"{pitch.achievable_angle(target):~.1f}")
    print(f"{'':>17}ground track runs {pitch.ground_distance(target):~.0f} meanwhile")
    print()

40 deg in 113 s:  needs 80.6 s, +40.1 % -- fits
                 largest slew that would fit: 65.0 deg
                 ground track runs 798 km meanwhile

40 deg in 90 s:  needs 80.6 s, +11.6 % -- fits
                 largest slew that would fit: 47.2 deg
                 ground track runs 635 km meanwhile



## Which spacecraft are we slewing?

The config names three inertia cases. `first_guess` estimates the inertia from a box and takes its mass from the mass budget, so it moves whenever the equipment list does. `measured_bol` and `measured_eol` state an inertia outright, as a mass properties report gives it, and need no mass at all.

The names are ours — nothing in the code matches on them. What matters is that a number can always be traced to the spacecraft that produced it.

In [8]:
cases = []
for name in config.inertia_cases:
    case = AgilityBudget(
        config, mission, Axis.PITCH, name, mass_budget=mass_data
    )
    cases.append({
        "case": name,
        "shape": "envelope" if case.case.is_envelope else "stated",
        "inertia": case.inertia.magnitude,
        "max_rate": case.max_rate().magnitude,
        "slew_40_deg": case.slew_time(40 * u.deg).magnitude,
        "achievable_in_113_s": case.achievable_angle(113 * u.s).magnitude,
    })

pd.DataFrame(cases).style.format(
    {"inertia": "{:,.1f}", "max_rate": "{:,.4f}",
     "slew_40_deg": "{:,.2f}", "achievable_in_113_s": "{:,.2f}"}
).hide(  # pyright: ignore[reportAttributeAccessIssue]
    axis="index"
)

case,shape,inertia,max_rate,slew_40_deg,achievable_in_113_s
first_guess,envelope,250.0,0.7727,60.65,65.00
measured_bol,stated,280.0,0.6899,66.86,58.04
measured_eol,stated,264.0,0.7317,63.54,61.55


## Which spacecraft are we slewing?

Here the cases stop agreeing with each other, and it matters more for pitch than it did for roll.

## The whole thing, in one table

`tabulated_agility()` lays the slew table out as a document. Given a target duration it also checks each angle against it and marks what does not fit; without one those columns are left out entirely, because there is nothing to check against.

## Pitch against roll

The appendage factor is the whole difference in this sample — 1.02 for pitch against 1.07 for roll — because the body is square in x and y, so the box term is identical for the two axes.

Read the sign carefully. The **envelope estimate makes pitch the better axis**: a smaller uplift means less inertia, so pitch turns faster and reaches further in the same time. The **measured cases say the opposite**, putting pitch some 6% heavier than roll.

Both cannot be right about the same spacecraft. An envelope factor is a guess at how the arrays and radiator load an axis; a mass properties report is a measurement. Where they disagree this sharply, the envelope's 1.02 is the number to doubt — and this table is the argument for measuring rather than estimating, which is what the cases were named for.

In [9]:
comparison = []
for name in config.inertia_cases:
    per_axis = {
        axis.value: AgilityBudget(config, mission, axis, name, mass_budget=mass_data)
        for axis in (Axis.ROLL, Axis.PITCH)
    }
    comparison.append({
        "case": name,
        "roll_inertia": per_axis["roll"].inertia.magnitude,
        "pitch_inertia": per_axis["pitch"].inertia.magnitude,
        "pitch_vs_roll": (
            per_axis["pitch"].inertia / per_axis["roll"].inertia
        ).magnitude - 1,
        "roll_40_deg": per_axis["roll"].slew_time(40 * u.deg).magnitude,
        "pitch_40_deg": per_axis["pitch"].slew_time(40 * u.deg).magnitude,
    })

pd.DataFrame(comparison).style.format(
    {"roll_inertia": "{:,.1f}", "pitch_inertia": "{:,.1f}",
     "pitch_vs_roll": "{:+.1%}", "roll_40_deg": "{:,.2f}",
     "pitch_40_deg": "{:,.2f}"}
).hide(  # pyright: ignore[reportAttributeAccessIssue]
    axis="index"
)

case,roll_inertia,pitch_inertia,pitch_vs_roll,roll_40_deg,pitch_40_deg
first_guess,262.3,250.0,-4.7%,63.19,60.65
measured_bol,264.7,280.0,+5.8%,63.69,66.86
measured_eol,250.0,264.0,+5.6%,60.65,63.54


In [10]:
print(f"inertia case  '{pitch.case_name}'  {pitch.inertia:~.1f}")
print()
pitch.tabulated_agility(target_duration=113 * u.s)

inertia case  'first_guess'  250.0 kg * m ** 2



Slew [deg],Limited by,Slew [s],With settling [s],Peak rate [deg/s],Momentum used,Time margin,
5,torque,15.2,35.2,0.6596,85.4%,+221.4%,PASS
10,momentum,21.8,41.8,0.7727,100.0%,+170.2%,PASS
15,momentum,28.3,48.3,0.7727,100.0%,+134.0%,PASS
20,momentum,34.8,54.8,0.7727,100.0%,+106.3%,PASS
40,momentum,60.6,80.6,0.7727,100.0%,+40.1%,PASS
45,momentum,67.1,87.1,0.7727,100.0%,+29.7%,PASS
60,momentum,86.5,106.5,0.7727,100.0%,+6.1%,PASS
90,momentum,125.4,145.4,0.7727,100.0%,-22.3%,FAILS


## One wheel failed

With one of four gone, only two of the three survivors can be driven at full torque if the net in-plane momentum is to stay zero. Both the momentum and the torque about the axis halve, so the rate halves and the acceleration halves with it.

This is the same pyramid flying degraded, not a three-wheel mounting: the geometry and the projection are unchanged, only the usable count drops.

In [11]:
nominal = pitch.slew_table()[["angle", "slew_time"]]
degraded = pitch.slew_table(degraded=True)[["angle", "slew_time"]]

comparison = nominal.merge(degraded, on="angle", suffixes=("_4_wheels", "_3_wheels"))
comparison["penalty"] = (
    comparison["slew_time_3_wheels"] / comparison["slew_time_4_wheels"] - 1
)
comparison.style.format(
    {"angle": "{:,.0f}", "slew_time_4_wheels": "{:,.1f}",
     "slew_time_3_wheels": "{:,.1f}", "penalty": "{:+.0%}"}
).hide(  # pyright: ignore[reportAttributeAccessIssue]
    axis="index"
)

angle,slew_time_4_wheels,slew_time_3_wheels,penalty
5,15.2,21.8,+44%
10,21.8,34.8,+59%
15,28.3,47.7,+69%
20,34.8,60.6,+74%
40,60.6,112.4,+85%
45,67.1,125.4,+87%
60,86.5,164.2,+90%
90,125.4,241.8,+93%
